Consigna:

Un grupo asesor financiero, integrado por expertos en IO de una consultora, ha sido contratado por el directorio de una compañía para ayudarle a maximizar el retorno sobre una inversión de U$S 10.000.000 (diez millones de dólares) en un portafolio constituido por cinco acciones específicas: Apple Inc. (AAPL), Microsoft Corp. (MSFT), Amazon.com Inc. (AMZN), Alphabet Inc. (GOOGL) y Tesla Inc. (TSLA).

El grupo asesor ha analizado los rendimientos esperados anuales de estas acciones: Apple Inc. ofrece un 12%, Microsoft Corp. un 10%, Amazon.com Inc. un 15%, Alphabet Inc. un 9% y Tesla Inc. un 11%. Para garantizar una adecuada diversificación y minimizar riesgos, el grupo asesor recomienda invertir al menos U$S 1.000.000 en Apple Inc., U$S 1.500.000 en Microsoft Corp., U$S 2.000.000 en Amazon.com Inc., U$S 500.000 en Alphabet Inc. y U$S 1.000.000 en Tesla Inc.

Además, el grupo asesor establece que la inversión conjunta en acciones de Apple Inc. (AAPL) y Microsoft Corp. (MSFT) no debe exceder el doble de la inversión en las acciones de Amazon.com Inc. (AMZN) y Alphabet Inc. (GOOGL), Por último, sugiere que la inversión en Apple Inc. (AAPL) no debe exceder la inversión en Microsoft Corp. (MSFT) en más de U$S 1.000.000

#### 1- Importación de librerías

In [207]:
import pulp as pu
from pulp import LpMaximize
from pulp import LpContinuous
from pulp import LpStatus
from pulp import value

#### 2- Creación del problema de programación lineal denominado "IOL"

In [208]:
problema = pu.LpProblem("IOL", LpMaximize)

#### 3- Definición de las variables de decisión

In [209]:
x1 = pu.LpVariable("x1", lowBound=0, cat=LpContinuous)
x2 = pu.LpVariable("x2", lowBound=0, cat=LpContinuous)
x3 = pu.LpVariable("x3", lowBound=0, cat=LpContinuous)
x4 = pu.LpVariable("x4", lowBound=0, cat=LpContinuous)
x5 = pu.LpVariable("x5", lowBound=0, cat=LpContinuous)

#### 4- Asignación de nuestras variables de decisión al problema creado inicialmente

#### acción	   rendimiento

##### APPL -> 0,12

##### MSFT	-> 0,1

##### AMZN0 -> 0,15

##### GOOGL	-> 0,9

##### TSLA	->   0,11


In [210]:
problema+= 0.12*x1+0.1*x2+0.15*x3+0.09*x4+0.11*x5, "Utilidad"

#### 5- Definimos las restricciones y las asignamos a nuestro problema

In [211]:
problema+= x1+x2+x3+x4+x5 == 10000000, "inversion_total"
problema+= x1  >= 1000000 , "umbral_inversion_appl"
problema+= x2   >= 1500000 , "umbral_inversion_msft"
problema+= x3  >= 2000000 , "umbral_inversion_amzn"
problema+= x4  >= 500000 , "umbral_inversion_googl"
problema+= x5  >= 1000000 , "umbral_inversion_tsla"
problema+= x1 + x2 <= (x3 + x4)*2 , "techo_inversion_appl_msft"
problema+= x1 <= x2 + 1000000 , "techo_inversión_appl"

In [212]:
problema

IOL:
MAXIMIZE
0.12*x1 + 0.1*x2 + 0.15*x3 + 0.09*x4 + 0.11*x5 + 0.0
SUBJECT TO
inversion_total: x1 + x2 + x3 + x4 + x5 = 10000000

umbral_inversion_appl: x1 >= 1000000

umbral_inversion_msft: x2 >= 1500000

umbral_inversion_amzn: x3 >= 2000000

umbral_inversion_googl: x4 >= 500000

umbral_inversion_tsla: x5 >= 1000000

techo_inversion_appl_msft: x1 + x2 - 2 x3 - 2 x4 <= 0

techo_inversión_appl: x1 - x2 <= 1000000

VARIABLES
x1 Continuous
x2 Continuous
x3 Continuous
x4 Continuous
x5 Continuous

#### 6- Resolución del problema

In [213]:
problema.solve()

1

In [214]:
LpStatus[problema.status]

'Optimal'

In [215]:
print("Los valores óptimos de las variables de decisión son:")
print("Monto total a invertir en APPL : ", x1.varValue)
print("Monto total a invertir en MSFT : ", x2.varValue) 
print("Monto total a invertir en AMZN : ", x3.varValue)
print("Monto total a invertir en GOOGL : ", x4.varValue)
print("Monto total a invertir en TSLA : ", x5.varValue)

print("El valor de la función objetivo evaluado en el punto óptimo es: ")
print("Utilidad máxima: ",value(problema.objective))

Los valores óptimos de las variables de decisión son:
Monto total a invertir en APPL :  1000000.0
Monto total a invertir en MSFT :  1500000.0
Monto total a invertir en AMZN :  6000000.0
Monto total a invertir en GOOGL :  500000.0
Monto total a invertir en TSLA :  1000000.0
El valor de la función objetivo evaluado en el punto óptimo es: 
Utilidad máxima:  1325000.0


In [216]:
### Creamos una lista el análisis de sensibilidad de las restricciones
sensibilidad = [{"Restricción":name, "Precio sombra":c.pi, "Holgura": c.slack} for name, c in problema.constraints.items()]

sensibilidad

[{'Restricción': 'inversion_total', 'Precio sombra': 0.15, 'Holgura': -0.0},
 {'Restricción': 'umbral_inversion_appl',
  'Precio sombra': -0.03,
  'Holgura': -0.0},
 {'Restricción': 'umbral_inversion_msft',
  'Precio sombra': -0.05,
  'Holgura': -0.0},
 {'Restricción': 'umbral_inversion_amzn',
  'Precio sombra': -0.0,
  'Holgura': -4000000.0},
 {'Restricción': 'umbral_inversion_googl',
  'Precio sombra': -0.06,
  'Holgura': -0.0},
 {'Restricción': 'umbral_inversion_tsla',
  'Precio sombra': -0.04,
  'Holgura': -0.0},
 {'Restricción': 'techo_inversion_appl_msft',
  'Precio sombra': -0.0,
  'Holgura': 10500000.0},
 {'Restricción': 'techo_inversión_appl',
  'Precio sombra': -0.0,
  'Holgura': 1500000.0}]